# 硬件与异步计算

本notebook介绍深度学习的硬件基础和异步计算机制。

## 学习目标

- 理解现代计算机硬件架构
- 掌握CPU、GPU、内存的性能特征
- 理解异步计算的工作原理
- 学习如何利用异步提升性能

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import time
import matplotlib.pyplot as plt

print(f"PyTorch版本: {torch.__version__}")
print(f"CUDA可用: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA版本: {torch.version.cuda}")
    print(f"GPU数量: {torch.cuda.device_count()}")
    print(f"当前GPU: {torch.cuda.get_device_name(0)}")

## 1. 计算机硬件基础

### 1.1 现代计算机组成

一台典型的深度学习工作站包含:

**核心组件**:
- **CPU**: 8-64核心,主频2-5GHz,控制整体流程
- **内存(RAM)**: 32-512GB DDR4/DDR5,带宽40-100GB/s
- **GPU**: 1-8个加速卡,数千个核心,专用显存
- **存储**: SSD/NVMe(500-7000MB/s) + HDD(100-200MB/s)
- **网络**: 1-100Gb以太网,用于分布式训练

**连接方式**:
- PCIe总线: CPU与GPU/存储的高速通道(16GB/s per PCIe 4.0 x16)
- 内存总线: CPU直连RAM
- NVLink: GPU之间的专用高速互连(600GB/s for A100)

### 1.2 关键性能指标

**延迟(Latency)**数字 - 每个程序员都应该知道:

| 操作 | 延迟 | 相对时间 |
|------|------|----------|
| L1缓存引用 | 0.5ns | 1秒 |
| L2缓存引用 | 7ns | 14秒 |
| 内存引用 | 100ns | 3分钟 |
| SSD随机读 | 150μs | 4天 |
| HDD寻道 | 10ms | 8个月 |
| 网络包:加州→荷兰 | 150ms | 10年 |

**关键启示**:
1. 缓存比内存快100倍
2. 内存比SSD快1000倍
3. SSD比HDD快100倍
4. 本地计算比网络传输快百万倍

In [ ]:
# 演示内存访问模式的影响
def test_memory_access():
    """测试顺序访问vs随机访问"""
    n = 10000000
    arr = np.arange(n, dtype=np.int32)
    
    # 顺序访问
    start = time.time()
    total = 0
    for i in range(0, n, 100):  # 每100个采样1个
        total += arr[i]
    sequential_time = time.time() - start
    
    # 随机访问
    indices = np.random.randint(0, n, size=n//100)
    start = time.time()
    total = 0
    for i in indices:
        total += arr[i]
    random_time = time.time() - start
    
    print(f"顺序访问时间: {sequential_time:.4f}秒")
    print(f"随机访问时间: {random_time:.4f}秒")
    print(f"随机访问慢 {random_time/sequential_time:.2f}倍")
    print("\n原因: 顺序访问利用了CPU缓存和预取机制")

test_memory_access()

## 2. CPU架构详解

### 2.1 CPU核心组件

现代CPU核心包含:

1. **前端(Frontend)**
   - 取指令(Instruction Fetch)
   - 分支预测(Branch Prediction)
   - 指令解码(Decode)

2. **执行引擎(Execution Engine)**
   - 整数单元(ALU)
   - 浮点单元(FPU)
   - 向量单元(SIMD: AVX2/AVX-512)

3. **缓存层次(Cache Hierarchy)**
   - L1缓存: 32-64KB, 延迟~4周期
   - L2缓存: 256-512KB, 延迟~12周期
   - L3缓存: 8-64MB(共享), 延迟~40周期

### 2.2 向量化(Vectorization)

**SIMD**(Single Instruction Multiple Data):
- 一条指令处理多个数据
- AVX2: 256位寄存器 → 8个float32或4个float64
- AVX-512: 512位寄存器 → 16个float32或8个float64

**例子**: 向量加法
```
标量: a[i] = b[i] + c[i]  (每次1个)
AVX2:  a[i:i+8] = b[i:i+8] + c[i:i+8]  (每次8个)
```

**加速比**: 理论上8倍(AVX2)或16倍(AVX-512)

In [ ]:
# 演示向量化的威力
def compare_vectorization():
    """对比NumPy向量化vs Python循环"""
    n = 10000000
    a = np.random.randn(n).astype(np.float32)
    b = np.random.randn(n).astype(np.float32)
    
    # NumPy向量化(调用BLAS,使用SIMD)
    start = time.time()
    c = a + b
    numpy_time = time.time() - start
    
    # Python循环(纯Python,无向量化)
    start = time.time()
    c_loop = [a[i] + b[i] for i in range(min(n, 100000))]  # 只测试部分避免太慢
    loop_time = (time.time() - start) * (n / 100000)  # 推算全部时间
    
    print(f"NumPy向量化: {numpy_time:.4f}秒")
    print(f"Python循环(推算): {loop_time:.4f}秒")
    print(f"向量化加速 {loop_time/numpy_time:.1f}倍")
    print("\n结论: 深度学习框架大量使用向量化操作!")

compare_vectorization()

## 3. GPU架构详解

### 3.1 GPU vs CPU

| 特性 | CPU | GPU |
|------|-----|-----|
| 核心数 | 8-64 | 2000-10000+ |
| 频率 | 2-5GHz | 1-2GHz |
| 设计目标 | 低延迟 | 高吞吐 |
| 缓存 | 大(MB级) | 小(KB级) |
| 线程切换 | 昂贵 | 廉价 |
| 适合任务 | 串行,分支多 | 并行,数据密集 |

### 3.2 GPU架构(以NVIDIA为例)

**层次结构**:
```
GPU
├── GPC (Graphics Processing Cluster) x N
│   ├── SM (Streaming Multiprocessor) x M
│   │   ├── CUDA Core x 64-128
│   │   ├── Tensor Core x 4-8 (Volta+)
│   │   ├── Shared Memory (64-128KB)
│   │   └── L1 Cache
│   └── ...
├── L2 Cache (几MB)
└── HBM/GDDR Memory (8-80GB)
```

**NVIDIA A100 (Ampere)示例**:
- 108个SM
- 6912个CUDA核心
- 432个Tensor核心
- 40/80GB HBM2显存
- 1555GB/s内存带宽
- 19.5 TFLOPS FP32, 312 TFLOPS Tensor

### 3.3 GPU内存

**类型**:
- **GDDR6**: 消费级GPU(RTX 3090: 24GB, 936GB/s)
- **HBM2/HBM2e**: 专业级GPU(A100: 80GB, 2TB/s)

**特点**:
- 带宽远高于系统内存(10倍+)
- 容量通常较小
- 成本高

In [ ]:
# GPU信息查询
def print_gpu_info():
    """打印GPU详细信息"""
    if not torch.cuda.is_available():
        print("没有可用的GPU")
        return
    
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"\n=== GPU {i}: {props.name} ===")
        print(f"计算能力: {props.major}.{props.minor}")
        print(f"总显存: {props.total_memory / 1024**3:.2f} GB")
        print(f"SM数量: {props.multi_processor_count}")
        print(f"最大线程/块: {props.max_threads_per_multi_processor}")
        print(f"Warp大小: {props.warp_size}")
        
        # 查询当前显存使用
        print(f"\n显存使用:")
        print(f"  已分配: {torch.cuda.memory_allocated(i) / 1024**3:.2f} GB")
        print(f"  已缓存: {torch.cuda.memory_reserved(i) / 1024**3:.2f} GB")

print_gpu_info()

## 4. 异步计算原理

### 4.1 为什么需要异步?

**同步执行的问题**:
```python
# Python线程
x = torch.randn(1000, 1000, device='cuda')
y = x + 1  # ← 等待GPU完成
z = y * 2  # ← 等待GPU完成
# Python线程被阻塞,CPU空闲!
```

**异步执行的优势**:
```python
# Python线程
x = torch.randn(1000, 1000, device='cuda')
y = x + 1  # ← 提交到GPU队列,立即返回
z = y * 2  # ← 提交到GPU队列,立即返回
# Python继续执行,GPU后台工作
```

### 4.2 前端与后端

**前端(Frontend)**:
- Python解释器
- 用户代码执行
- 快速提交任务到后端

**后端(Backend)**:
- C++实现
- 管理计算图
- 调度GPU核函数(kernels)
- 实际执行计算

### 4.3 依赖关系跟踪

后端维护计算图,自动处理依赖:
```python
a = torch.randn(100, 100, device='cuda')
b = torch.randn(100, 100, device='cuda')  # 并行
c = a + b  # 等待a和b完成
d = c * 2  # 等待c完成
```

**计算图**:
```
    a      b
     \    /
       c
       |
       d
```

In [ ]:
# 演示异步计算
def demo_async_computation():
    """对比同步vs异步执行"""
    if not torch.cuda.is_available():
        print("需要GPU来演示异步计算")
        return
    
    device = torch.device('cuda')
    n = 5000
    
    # 预热GPU
    _ = torch.randn(n, n, device=device)
    
    print("=== 异步执行(默认) ===")
    start = time.time()
    for _ in range(100):
        a = torch.randn(n, n, device=device)
        b = torch.mm(a, a)
    async_time = time.time() - start
    print(f"异步时间(仅提交): {async_time:.4f}秒")
    
    # 强制同步
    print("\n=== 强制同步执行 ===")
    start = time.time()
    for _ in range(100):
        a = torch.randn(n, n, device=device)
        b = torch.mm(a, a)
        torch.cuda.synchronize()  # 等待GPU完成
    sync_time = time.time() - start
    print(f"同步时间(实际执行): {sync_time:.4f}秒")
    
    print(f"\n观察: 异步'时间'很短是因为只是提交任务,没等GPU执行完")
    print(f"实际GPU执行时间: {sync_time:.4f}秒")

demo_async_computation()

## 5. 异步计算的实践

### 5.1 CPU-GPU数据传输

**同步传输**:
```python
# 计算完成 → 传输开始
y_gpu = compute_on_gpu(x)
y_cpu = y_gpu.to('cpu')  # 阻塞直到GPU完成
```

**异步传输**(重叠计算与通信):
```python
# 计算和传输并行
results = []
for x in data:
    y = compute_on_gpu(x)
    y_cpu = y.to('cpu', non_blocking=True)  # 不等待
    results.append(y_cpu)
torch.cuda.synchronize()  # 最后统一同步
```

### 5.2 多流(Streams)

**默认流**: 所有操作串行
**多个流**: 不同流中的操作可并行

```python
stream1 = torch.cuda.Stream()
stream2 = torch.cuda.Stream()

with torch.cuda.stream(stream1):
    output1 = model1(input1)  # 流1

with torch.cuda.stream(stream2):
    output2 = model2(input2)  # 流2,与流1并行
```

### 5.3 同步点

**隐式同步**(自动触发):
- `.item()`, `.numpy()`: 转到Python标量/NumPy
- 打印tensor值
- CPU-GPU数据拷贝(默认)
- 某些内存操作

**显式同步**:
```python
torch.cuda.synchronize()  # 等待当前设备所有流
torch.cuda.synchronize(device=0)  # 等待指定设备
```

In [ ]:
# 演示计算与通信重叠
def demo_compute_transfer_overlap():
    """演示异步传输的性能提升"""
    if not torch.cuda.is_available():
        print("需要GPU")
        return
    
    device = torch.device('cuda')
    n_batches = 50
    size = (1000, 1000)
    
    # 方法1: 同步传输(计算→传输→计算→传输)
    print("=== 同步传输 ===")
    start = time.time()
    for _ in range(n_batches):
        x = torch.randn(size, device=device)
        y = torch.mm(x, x)  # GPU计算
        y_cpu = y.to('cpu')  # 传输(等待计算完成)
    torch.cuda.synchronize()
    sync_time = time.time() - start
    print(f"时间: {sync_time:.4f}秒")
    
    # 方法2: 异步传输(计算与传输重叠)
    print("\n=== 异步传输(non_blocking=True) ===")
    start = time.time()
    for _ in range(n_batches):
        x = torch.randn(size, device=device)
        y = torch.mm(x, x)  # GPU计算
        y_cpu = y.to('cpu', non_blocking=True)  # 异步传输
    torch.cuda.synchronize()
    async_time = time.time() - start
    print(f"时间: {async_time:.4f}秒")
    
    print(f"\n加速比: {sync_time/async_time:.2f}x")
    print("原理: 当前批次传输时,下一批次已在GPU上计算")

demo_compute_transfer_overlap()

## 6. 性能优化建议

### 6.1 内存访问优化

**DO**:
- ✅ 顺序访问内存(利用缓存)
- ✅ 批处理操作(减少函数调用开销)
- ✅ 使用连续内存(`.contiguous()`)
- ✅ 预分配内存(避免频繁分配)

**DON'T**:
- ❌ 随机访问
- ❌ 小操作循环
- ❌ 频繁的内存分配/释放
- ❌ 不必要的数据拷贝

### 6.2 异步编程最佳实践

1. **最小化同步点**
   ```python
   # Bad: 频繁同步
   for x in data:
       y = model(x)
       print(y.item())  # 每次都同步!
   
   # Good: 批量同步
   results = [model(x) for x in data]
   torch.cuda.synchronize()
   for y in results:
       print(y.item())
   ```

2. **使用non_blocking传输**
   ```python
   data = data.to(device, non_blocking=True)
   ```

3. **Pin内存加速传输**
   ```python
   loader = DataLoader(dataset, pin_memory=True)
   ```

4. **多流并行**
   ```python
   streams = [torch.cuda.Stream() for _ in range(n)]
   with torch.cuda.stream(streams[i % n]):
       process(data[i])
   ```

### 6.3 性能分析工具

**PyTorch Profiler**:
```python
from torch.profiler import profile, ProfilerActivity

with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA]) as prof:
    model(input)

print(prof.key_averages().table(sort_by="cuda_time_total"))
```

**NVIDIA工具**:
- `nvidia-smi`: GPU使用监控
- `nvprof`: 性能分析
- `Nsight Systems/Compute`: 深度分析

In [ ]:
# 完整示例: 优化的训练循环
class OptimizedTraining:
    """展示异步计算的最佳实践"""
    
    def __init__(self, model, device='cuda'):
        self.model = model.to(device)
        self.device = device
    
    def train_step_naive(self, data_loader, optimizer):
        """朴素实现(多个同步点)"""
        total_loss = 0
        for batch in data_loader:
            data, target = batch
            data = data.to(self.device)  # 同步
            target = target.to(self.device)  # 同步
            
            output = self.model(data)
            loss = nn.functional.cross_entropy(output, target)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()  # 同步!
        
        return total_loss / len(data_loader)
    
    def train_step_optimized(self, data_loader, optimizer):
        """优化实现(减少同步)"""
        losses = []
        
        for batch in data_loader:
            data, target = batch
            # 异步传输
            data = data.to(self.device, non_blocking=True)
            target = target.to(self.device, non_blocking=True)
            
            output = self.model(data)
            loss = nn.functional.cross_entropy(output, target)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            losses.append(loss)  # 不同步,保存tensor
        
        # 批量同步
        torch.cuda.synchronize()
        return sum(l.item() for l in losses) / len(losses)

print("优化要点:")
print("1. 使用non_blocking=True异步传输数据")
print("2. 避免在循环中调用.item()(会同步)")
print("3. DataLoader设置pin_memory=True")
print("4. 批量结束后统一同步")

## 7. 小结

### 核心要点

1. **硬件理解**
   - CPU: 低延迟,强大单核,大缓存
   - GPU: 高吞吐,数千核心,高带宽显存
   - 内存层次: 寄存器 > L1 > L2 > L3 > RAM > SSD > HDD

2. **性能关键**
   - 缓存局部性(spatial & temporal)
   - 向量化(SIMD)
   - 并行化(多核/多GPU)
   - 减少内存传输

3. **异步计算**
   - 前端快速提交,后端执行
   - 重叠计算与通信
   - 最小化同步点
   - 利用non_blocking传输

### 性能优化checklist

- [ ] 使用GPU加速
- [ ] 批处理数据(增大batch size)
- [ ] pin_memory=True in DataLoader
- [ ] non_blocking=True in .to()
- [ ] 避免频繁CPU-GPU传输
- [ ] 减少.item()/.numpy()调用
- [ ] 使用混合精度训练(FP16)
- [ ] 多GPU并行(后续章节)

### 推荐资源

- [PyTorch性能调优指南](https://pytorch.org/tutorials/recipes/recipes/tuning_guide.html)
- [NVIDIA深度学习性能指南](https://docs.nvidia.com/deeplearning/performance/)
- [计算机体系结构量化研究方法](https://www.elsevier.com/books/computer-architecture/hennessy/978-0-12-383872-8)

## 练习

1. **内存层次实验**: 测量访问不同大小数组的速度(10KB, 100KB, 1MB, 10MB, 100MB),观察缓存效应。

2. **向量化对比**: 用纯Python循环和NumPy实现矩阵乘法,对比性能。

3. **异步传输**: 实现同步和异步数据传输,测量性能差异。

4. **GPU利用率**: 使用`nvidia-smi`监控训练过程的GPU利用率,分析瓶颈。

5. **Profiling**: 使用PyTorch Profiler分析模型的性能热点。